In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/figure-1"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import scanpy as sc

from spatial_tcr.clonal_expansion import annotate_singlets, get_avbv_expression
from spatial_tcr.tcr import get_tcr_genes

In [2]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/06.2-kidney_tcr_filtered.h5ad"
adata = sc.read_h5ad(path)
adata

AnnData object with n_obs × n_vars = 510139 × 431
    obs: 'sample', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'condition', 'cc', 'cell_type_no_tcr', 'cell_type_no_tcr_prob', 'tcell_subtype', 'cell_type_l1', 'cell_type_l2', 'is_ATL', 'is_B', 'is_CNT', 'is_DCT', 'is_DTL', 'is_EC', 'is_FIB', 'is_IC', 'is_MAST', 'is_MC', 'is_MDC', 'is_Mac', 'is_N', 'is_NEU', 'is_PC', 'is_PEC', 'is_PL', 'is_POD', 'is_PT', 'is_PapE', 'is_T', 'is_TAL', 'is_VSM/P', 'is_cDC', 'is_cycMNP', 'is_glom. EC', 'is_pDC', 'is_unknown', 'leiden', 'glom_annot', 'in_glom', 'tcell_density_group', 'tcell_density', 'tcell_infiltrate'
    var: 'n_cells_by_co

In [3]:
adata.obs["cell_type_l1.1"] = adata.obs["cell_type_l2"].astype(str)
adata.obs["cell_type_l1.1"] = adata.obs["cell_type_l1.1"].replace({"Tregs": "CD4+"})
adata.obs["cell_type_l1.1"].value_counts()

cell_type_l1.1
PT          120985
FIB          79706
Mac          51415
TAL          46477
EC           41864
MDC          21738
PC           19094
IC           18190
VSM/P        16335
CNT          14542
DCT          14132
CD4+         10416
POD           8584
glom. EC      6554
CD8+          6509
PL            6156
B             5862
DTL           5230
PEC           4937
MC            2816
unknown       2241
NKT-like      1842
MAST          1154
MAIT           820
cycMNP         771
N              544
gdT            446
ATL            396
cDC            303
pDC             37
PapE            27
NEU             16
Name: count, dtype: int64

In [4]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)
annotate_singlets(adata, genes=av_genes, out_key="av_clone", allow_multiplets=True)
annotate_singlets(adata, genes=bv_genes, out_key="bv_clone", allow_multiplets=True)

Found 35 TRAV genes, 31 TRBV genes, 3 TRDV genes, 14 TRGV genes
Found 58491 singlets
Found 9982 multiplets
Found 1377 valid multiplets
Found 74244 singlets
Found 12914 multiplets
Found 1473 valid multiplets


In [5]:
adata.obs["bv_clone"].value_counts(dropna=True).sum()

np.int64(75717)

In [6]:
adata.obs["bv_clone"].value_counts(dropna=True)

bv_clone
TRBV5-6      18337
TRBV6         9531
TRBV10        5449
TRBV7-2_3     2869
TRBV7-6       2848
TRBV18_19     2773
TRBV11        2462
TRBV4         2351
TRBV28        2132
TRBV12        2101
TRBV21-1      1931
TRBV27        1840
TRBV24-1      1657
TRBV5-1       1634
TRBV14        1553
TRBV16        1480
TRBV12-1      1294
TRBV5-2       1282
TRBV8-1       1278
TRBV23-1      1274
TRBV30        1215
TRBV7-9       1143
TRBVB         1127
TRBV9         1037
TRBV25-1      1024
TRBV15        1016
TRBV3-1        804
TRBV1          793
TRBV13         713
TRBV11-3       534
TRBV6-6        235
Name: count, dtype: int64

In [7]:
adata.obs["avbv_clone"] = (
    adata.obs["av_clone"].astype(str) + "+" + adata.obs["bv_clone"].astype(str)
)
# remove None values
adata.obs.loc[adata.obs["av_clone"].isna(), "avbv_clone"] = None
adata.obs.loc[adata.obs["bv_clone"].isna(), "avbv_clone"] = None
adata.obs["avbv_clone"].value_counts(dropna=True).sum()

np.int64(14735)

In [8]:
df_avbv = get_avbv_expression(adata, layer="counts")
df_avbv.head()

Found 35 TRAV genes, 31 TRBV genes, 3 TRDV genes, 14 TRGV genes
Found 35 TRAV genes, 31 TRBV genes, 3 TRDV genes, 14 TRGV genes


,TRAV1-1+TRBV1,TRAV1-1+TRBV11-3,TRAV1-1+TRBV12-1,TRAV1-1+TRBV13,TRAV1-1+TRBV14,TRAV1-1+TRBV15,TRAV1-1+TRBV16,TRAV1-1+TRBV21-1,TRAV1-1+TRBV23-1,TRAV1-1+TRBV24-1,...,TRAV9+TRBVB,TRAV9+TRBV10,TRAV9+TRBV11,TRAV9+TRBV12,TRAV9+TRBV18_19,TRAV9+TRBV4,TRAV9+TRBV5-6,TRAV9+TRBV6,TRAV9+TRBV7-2_3,TRAV9+TRBV7-6
aaaombci-1output-XETG00088__0029040__Region_2__20240719__095641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aaapplje-1output-XETG00088__0029040__Region_2__20240719__095641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aabafpch-1output-XETG00088__0029040__Region_2__20240719__095641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aabcmlje-1output-XETG00088__0029040__Region_2__20240719__095641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
aabecado-1output-XETG00088__0029040__Region_2__20240719__095641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
adata.write_h5ad(f"{data_dir}/07.1-kidney_tcr_clones.h5ad")

/home/dschaub/.uv-local/venvs/xenium-tcr/lib/python3.13/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)
